# 01 — 대전 탐색 및 파이프라인 수립위성영상으로 산림 유형을 분류할 수 있는지 **한 지역에서 먼저 확인**한 노트북입니다.여기서 검증한 절차를 `02`에서 함수로 묶어 세 지역에 적용합니다.| 단계 | 셀 | 내용 ||---|---|---|| 준비 | `[A]` | 라이브러리 · GEE 인증 || 예비 확인 | 홍천 | 낙엽기에 신호가 생기는지 눈으로 확인 || 데이터 생성 | `[B]`~`[I]` | AOI 추출 → 위성 내려받기 → 라벨 래스터화 || 정합 검증 | `[J]`~`[J3]` | 영상과 라벨이 맞는지 · **낙엽기 신호 정량 확인** || 모델 | `[K]`~`[P2]` | 공간 분할 → 피처 → 학습 → 개선 || 오차 분석 | `[Q]`~`[U]` | 혼효림 진단 · 탄소 오차 · **공간 분리 검증** |> 탄소계수는 임시값 55/65/60 입니다. 확정 계수는 `03` 참조.---## 준비

In [ ]:
# ============================================================
# [A] 셋업 + 유틸  ← 런타임 끊기면 이 셀부터 실행
# ============================================================
!pip install -q earthengine-api geemap geopandas pyogrio

import ee, geemap, pandas as pd, geopandas as gpd

PROJECT_ID = 'driven-era-467023-m3'
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SHP = '/content/drive/MyDrive/forest/data/imsang/2025_daejeon/30.shp'
gdf = gpd.read_file(SHP, engine='pyogrio')

def cloud_mask(img):
    """SCL 밴드로 구름/그림자 제거 후 반사율 0~1 스케일 변환"""
    scl = img.select('SCL')
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good).divide(10000)

def composite_aoi(aoi, start, end, months, cloud_pct=50):
    """지정 AOI/기간/월의 Sentinel-2를 median 합성"""
    mf = ee.Filter.Or([ee.Filter.calendarRange(m, m, 'month') for m in months])
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi).filterDate(start, end).filter(mf)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_pct))
            .map(cloud_mask).median().clip(aoi))

print('셋업 완료 | 폴리곤', len(gdf))

In [ ]:
# 홍천군 일대 (나중에 이 값만 바꾸면 다른 지역)
AOI = ee.Geometry.Rectangle([127.75, 37.60, 128.15, 37.85])

def cloud_mask(img):
    scl = img.select('SCL')
    # 3=그림자, 8,9=구름, 10=권운
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good).divide(10000)

def composite(start, end):
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(AOI)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
            .map(cloud_mask)
            .median()
            .clip(AOI))

summer  = composite('2023-07-01', '2023-08-31')   # 여름
leafoff = composite('2023-11-15', '2023-12-31')   # 낙엽기

print('합성 완료')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


합성 완료


대전


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
d = '/content/drive/MyDrive/forest/data/imsang/2025_daejeon'
for f in sorted(os.listdir(d)):
    mb = os.path.getsize(os.path.join(d, f)) / 1e6
    print(f'{f:15s} {mb:8.1f} MB')

Mounted at /content/drive
30.dbf              55.1 MB
30.prj               0.0 MB
30.sbn               0.2 MB
30.sbx               0.0 MB
30.shp             101.0 MB
30.shx               0.2 MB


In [ ]:
!pip install -q geopandas pyogrio

import geopandas as gpd, pyogrio

SHP = f'{d}/30.shp'

info = pyogrio.read_info(SHP)
print('좌표계:', info['crs'])
print('폴리곤 수:', f"{info['features']:,}")
print()
print('컬럼 목록:')
for c in info['fields']:
    print(' ', c)

좌표계: EPSG:5179
폴리곤 수: 18,908

컬럼 목록:
  STORUNST
  FROR_CD
  FRTP_CD
  KOFTR_GROU
  DMCLS_CD
  AGCLS_CD
  DNST_CD
  HEIGHT
  LDMARK_STN
  MAP_LABEL
  갱신년도
  ETC_PCMTT
  FRTP_NM
  KOFTR_NM
  DMCLS_NM
  AGCLS_NM
  DNST_NM
  HEIGHT_NM
  Shape_Leng
  Shape_Area


In [ ]:
gdf = gpd.read_file(SHP, engine='pyogrio')
print(gdf.shape)

for col in ['FRTP_CD', 'STORUNST', 'FROR_CD', 'DNST_CD', 'AGCLS_CD', '갱신년도']:
    print(f'\n[{col}]')
    print(gdf[col].value_counts(dropna=False).to_string())

(18908, 21)

[FRTP_CD]
FRTP_CD
2    7362
1    5280
0    3307
3    2852
4     107

[STORUNST]
STORUNST
1    15601
2     3307

[FROR_CD]
FROR_CD
2    11113
1     4488
0     3307

[DNST_CD]
DNST_CD
C       13176
None     3414
B        1950
A         368

[AGCLS_CD]
AGCLS_CD
5       8174
4       5542
None    3414
3        991
2        376
1        216
6        179
8         10
7          5
9          1

[갱신년도]
갱신년도
2025    5641
2015    4883
2024    4193
None    2050
2020     908
2016     651
2018     401
2019      69
2017      52
2022      24
2021      19
2023      17


In [ ]:
gdf['area_ha'] = gdf.geometry.area / 10000   # 5179는 미터 단위

tab = (gdf.groupby(['FRTP_CD','FRTP_NM'])['area_ha']
         .agg(['count','sum']).reset_index())
tab['비율%'] = (tab['sum'] / tab['sum'].sum() * 100).round(1)
print(tab.to_string(index=False))

FRTP_CD  FRTP_NM  count          sum  비율%
      0 무립목지/비산림   3307  1079.631426  4.0
      1     침엽수림   5280  8900.691088 32.8
      2     활엽수림   7362 12517.692681 46.1
      3      혼효림   2852  4632.552841 17.1
      4       죽림    107    17.725829  0.1


---## 데이터 생성 — 대전좌표를 손으로 찍지 않고 **임상도 폴리곤 경계를 그대로** 촬영 범위로 씁니다.영상 범위와 라벨 범위가 자동으로 일치합니다.

In [ ]:
# ============================================================
# [B] 대전 AOI 생성
#   - 직접 좌표를 찍지 않고 임상도 폴리곤의 전체 경계를 그대로 사용
#   - 이렇게 하면 위성영상 범위와 라벨 범위가 자동으로 일치한다
#   - GEE는 위경도(EPSG:4326)만 받으므로 5179 -> 4326 변환 필요
# ============================================================
bounds = gdf.to_crs(4326).total_bounds        # [minx, miny, maxx, maxy]
AOI_DJ = ee.Geometry.Rectangle(list(bounds))

print('대전 경계(위경도):', bounds.round(4))

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


대전 경계(위경도): [127.2469  36.1831 127.5597  36.4988]


In [ ]:
# ============================================================
# [C] 대전 다시기 합성
#   - 임상도 갱신년도 최빈값이 2024~2025 이므로 영상도 같은 시기로 맞춤
#   - 2년치를 모아 구름 없는 픽셀 수를 확보 (한국 겨울은 맑은 날 영상이 귀함)
# ============================================================
summer_dj  = composite_aoi(AOI_DJ, '2024-01-01', '2025-12-31',
                           months=[6, 7, 8, 9])      # 생육기
leafoff_dj = composite_aoi(AOI_DJ, '2024-01-01', '2025-12-31',
                           months=[11, 12, 1, 2])    # 낙엽기

# 각 시기에 실제로 몇 장이 쓰였는지 확인 (3장 미만이면 품질 의심)
for nm, mo in [('여름', [6,7,8,9]), ('낙엽기', [11,12,1,2])]:
    mf = ee.Filter.Or([ee.Filter.calendarRange(m, m, 'month') for m in mo])
    n = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(AOI_DJ).filterDate('2024-01-01', '2025-12-31')
         .filter(mf).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50))
         .size().getInfo())
    print(f'{nm}: {n}장')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


여름: 23장
낙엽기: 31장


In [ ]:
# ============================================================
# [E] Drive로 내보내기
#   - 여름 10밴드 + 낙엽기 10밴드 = 총 20밴드 스택
#   - crs를 임상도와 동일한 EPSG:5179 로 지정하는 것이 핵심.
#     여기서 맞춰두면 나중에 라벨 래스터화 때 재투영이 필요 없다.
#   - 실행 후 code.earthengine.google.com 의 Tasks 탭에서 진행상황 확인
# ============================================================
BANDS = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']

stack = (summer_dj.select(BANDS).rename([f's_{b}' for b in BANDS])
         .addBands(leafoff_dj.select(BANDS).rename([f'w_{b}' for b in BANDS]))
         .toFloat())

print('밴드 수:', len(stack.bandNames().getInfo()))   # 20 이어야 정상

task = ee.batch.Export.image.toDrive(
    image          = stack,
    description    = 'daejeon_s2_stack',
    folder         = 'forest_s2',              # Drive 최상위에 자동 생성
    fileNamePrefix = 'daejeon_s2_20band',
    region         = AOI_DJ,
    scale          = 10,                       # Sentinel-2 해상도
    crs            = 'EPSG:5179',              # 임상도와 동일
    maxPixels      = 1e10
)
task.start()
print('export 시작됨. 10~30분 소요 예상')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


밴드 수: 20
export 시작됨. 10~30분 소요 예상


In [ ]:
# ============================================================
# [G0] export 결과 위치 찾기
#   - GEE는 Drive 최상위에 folder= 이름으로 생성한다
#   - 웹 UI 반영이 늦을 수 있으므로 파일시스템에서 직접 확인
# ============================================================
import glob, os

root = '/content/drive/MyDrive'
print('내 드라이브 최상위 폴더:')
for d in sorted(os.listdir(root)):
    if os.path.isdir(f'{root}/{d}'):
        print(' ', d)

print('\ndaejeon 이름이 들어간 tif 전체 검색:')
for p in glob.glob(f'{root}/**/*daejeon*.tif', recursive=True):
    print(f'  {p}  ({os.path.getsize(p)/1e6:.0f} MB)')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


내 드라이브 최상위 폴더:
  Classroom
  Colab Notebooks
  DL2025
  DS2025
  ML2025
  forest
  forest_s2
  termproject

daejeon 이름이 들어간 tif 전체 검색:
  /content/drive/MyDrive/forest_s2/daejeon_s2_20band.tif  (683 MB)


In [ ]:
# ============================================================
# [G0-2] export 결과를 프로젝트 폴더로 이동
#   - GEE는 항상 Drive 최상위에 저장하므로 수동 정리 필요
#   - shutil.move 는 같은 파일시스템 내 이동이라 복사보다 빠르다
# ============================================================
import shutil, os

SRC = '/content/drive/MyDrive/forest_s2/daejeon_s2_20band.tif'
DST_DIR = '/content/drive/MyDrive/forest/data/s2'
os.makedirs(DST_DIR, exist_ok=True)

S2_PATH = f'{DST_DIR}/daejeon_s2_20band.tif'
shutil.move(SRC, S2_PATH)
print('이동 완료:', S2_PATH, f'({os.path.getsize(S2_PATH)/1e6:.0f} MB)')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


이동 완료: /content/drive/MyDrive/forest/data/s2/daejeon_s2_20band.tif (683 MB)


In [ ]:
# ============================================================
# [G] 영상 메타데이터 확인
# ============================================================
!pip install -q rasterio
import rasterio

with rasterio.open(S2_PATH) as src:
    print('좌표계 :', src.crs)          # EPSG:5179
    print('밴드 수 :', src.count)        # 20
    print('크기   :', src.width, 'x', src.height)
    print('해상도 :', src.res)           # (10.0, 10.0)
    print('범위   :', [round(v) for v in src.bounds])
    print('밴드명 :', src.descriptions[:4])

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


좌표계 : EPSG:5179
밴드 수 : 20
크기   : 2813 x 3506
해상도 : (10.0, 10.0)
범위   : [977240, 1798440, 1005370, 1833500]
밴드명 : ('s_B2', 's_B3', 's_B4', 's_B5')


In [ ]:
# ============================================================
# [H] 임상도 → 라벨 래스터
#   - 위성영상의 transform/width/height 를 그대로 사용해
#     두 데이터가 픽셀 단위로 정확히 정렬되도록 한다
#   - 라벨과 갱신년도를 따로 만들어, 나중에
#     "2024년 이후 폴리곤만 학습" 같은 필터링이 가능하게 함
#   - FRTP_CD 는 VARCHAR 이므로 반드시 문자열('1','2'...)로 매핑
# ============================================================
import numpy as np, pandas as pd
from rasterio.features import rasterize

CLASS_MAP = {
    '0': 0,   # 무립목지/비산림 → 배경
    '1': 1,   # 침엽수림
    '2': 2,   # 활엽수림
    '3': 3,   # 혼효림
    '4': 0,   # 죽림 → 17.7ha뿐이라 배경 흡수
}

with rasterio.open(S2_PATH) as src:
    meta      = src.meta.copy()
    transform = src.transform
    shape     = (src.height, src.width)
    s2_crs    = src.crs

# 좌표계 안전장치
g = gdf.to_crs(s2_crs) if gdf.crs != s2_crs else gdf.copy()
print('좌표계:', g.crs)

# ---- 클래스 래스터 ----
g['cls'] = g['FRTP_CD'].map(CLASS_MAP).fillna(0).astype('uint8')
label = rasterize(
    ((geom, v) for geom, v in zip(g.geometry, g['cls'])),
    out_shape=shape, transform=transform,
    fill=255,            # 폴리곤 없는 영역 = nodata
    dtype='uint8'
)

# ---- 갱신년도 래스터 ----
g['yr'] = pd.to_numeric(g['갱신년도'], errors='coerce').fillna(0).astype('uint16')
year = rasterize(
    ((geom, v) for geom, v in zip(g.geometry, g['yr'])),
    out_shape=shape, transform=transform,
    fill=0, dtype='uint16'
)

# ---- 분포 확인 ----
names = {0:'배경', 1:'침엽수림', 2:'활엽수림', 3:'혼효림', 255:'nodata'}
u, c = np.unique(label, return_counts=True)
print()
for v, n in zip(u, c):
    print(f'{names.get(v, v):8s} {n:>12,} px  ({n*100/label.size:5.1f}%)')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


좌표계: EPSG:5179

배경            109,617 px  (  1.1%)
침엽수림          889,889 px  (  9.0%)
활엽수림        1,252,252 px  ( 12.7%)
혼효림           463,085 px  (  4.7%)
nodata      7,147,535 px  ( 72.5%)


In [ ]:
# ============================================================
# [I] 라벨 래스터 저장
#   - label: 0=배경 1=침엽 2=활엽 3=혼효 255=nodata
#   - year : 폴리곤 갱신년도 (0=미상)
#   - lzw 압축으로 용량 절약
# ============================================================
OUT = '/content/drive/MyDrive/forest/data'

for arr, name, dt, nod in [(label, 'daejeon_label', 'uint8', 255),
                           (year,  'daejeon_year',  'uint16', 0)]:
    m2 = meta.copy()
    m2.update(count=1, dtype=dt, nodata=nod, compress='lzw')
    path = f'{OUT}/{name}.tif'
    with rasterio.open(path, 'w', **m2) as dst:
        dst.write(arr, 1)
    print('저장:', path, f'({os.path.getsize(path)/1e6:.1f} MB)')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


저장: /content/drive/MyDrive/forest/data/daejeon_label.tif (0.7 MB)
저장: /content/drive/MyDrive/forest/data/daejeon_year.tif (1.3 MB)


---## 정합 검증영상 격자와 라벨 격자가 어긋나지 않았는지 확인합니다.

In [ ]:
# ============================================================
# [J] 정합 검증
#   - 위성 RGB 위에 라벨을 반투명으로 겹쳐 어긋남 확인
#   - 라벨 경계가 산 능선/계곡을 따라가면 정상
#   - 한쪽으로 밀려 있으면 좌표계 또는 transform 문제
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# 라벨이 많은 지역을 자동으로 찾아 그 주변을 잘라낸다
valid = np.argwhere(label != 255)
cy, cx = valid.mean(axis=0).astype(int)
sz = 1200
r0 = max(0, min(cy - sz//2, shape[0] - sz))
c0 = max(0, min(cx - sz//2, shape[1] - sz))

with rasterio.open(S2_PATH) as src:
    win = rasterio.windows.Window(c0, r0, sz, sz)
    rgb = src.read([3, 2, 1], window=win).astype('float32')   # B4,B3,B2
rgb = np.clip((rgb - 0.02) / 0.23, 0, 1).transpose(1, 2, 0)

lab   = label[r0:r0+sz, c0:c0+sz]
lab_m = np.ma.masked_where(lab == 255, lab)
cmap  = ListedColormap(['#bbbbbb', '#1b7837', '#d95f02', '#e7c419'])

fig, ax = plt.subplots(1, 3, figsize=(21, 7))
ax[0].imshow(rgb);                                ax[0].set_title('Sentinel-2 RGB')
ax[1].imshow(lab_m, cmap=cmap, vmin=0, vmax=3);   ax[1].set_title('label')
ax[2].imshow(rgb)
ax[2].imshow(lab_m, cmap=cmap, vmin=0, vmax=3, alpha=0.45)
ax[2].set_title('overlay | 회색=배경 녹색=침엽 주황=활엽 노랑=혼효')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


<Figure size 2100x700 with 3 Axes>

[그림/지도 출력 생략 — 파일 경량화]


In [ ]:
# ============================================================
# [J2] 정합 정량 검증
#   - 라벨된 산림 픽셀의 여름 NDVI가 높아야 정상
#   - 배경(비산림)과 뚜렷이 갈리면 위치 정합이 맞다는 뜻
# ============================================================
with rasterio.open(S2_PATH) as src:
    nir = src.read(7).astype('float32')    # s_B8
    red = src.read(3).astype('float32')    # s_B4
ndvi_s = (nir - red) / (nir + red + 1e-6)

print('클래스별 여름 NDVI 중앙값')
for v, n in [(0,'배경'), (1,'침엽수림'), (2,'활엽수림'), (3,'혼효림')]:
    m = (label == v)
    print(f'  {n:8s} {np.nanmedian(ndvi_s[m]): .3f}   ({m.sum():,} px)')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


클래스별 여름 NDVI 중앙값
  배경        0.820   (109,617 px)
  침엽수림      0.852   (889,889 px)
  활엽수림      0.885   (1,252,252 px)
  혼효림       0.873   (463,085 px)


### `[J3]` 낙엽기 신호 확인 — 모델을 만들기 *전에* 한 검증여름 단독으로는 세 클래스가 0.852~0.885로 뭉쳐 있고(폭 0.033), 낙엽기에는 0.438~0.646으로 벌어집니다(폭 0.208). **약 6배.**이걸 먼저 해둔 덕분에 이후 성능이 안 나올 때 "데이터 탓인가 모델 탓인가"를 구분할 수 있었습니다.

### `[J3]` 낙엽기 신호 확인 — 모델을 만들기 *전에* 한 검증여름만으로는 세 클래스가 뭉쳐 있고, 낙엽기에 벌어집니다.**데이터에 신호가 실재하는지 먼저 확인**해 두면, 나중에 성능이 안 나올 때"데이터 문제인가 모델 문제인가"를 구분할 수 있습니다.

In [ ]:
# ============================================================
# [J3] 낙엽기 신호 확인
#   - 여름만으로는 클래스가 안 갈린다는 것을 확인했으므로
#     낙엽기 NDVI와 dNDVI에서 분리가 일어나는지 검증
#   - 침엽 > 혼효 > 활엽 순서가 나와야 다시기 접근이 정당화된다
# ============================================================
with rasterio.open(S2_PATH) as src:
    w_nir = src.read(17).astype('float32')   # w_B8
    w_red = src.read(13).astype('float32')   # w_B4
ndvi_w = (w_nir - w_red) / (w_nir + w_red + 1e-6)
dndvi  = ndvi_s - ndvi_w

print(f"{'클래스':10s}{'여름':>8s}{'낙엽기':>9s}{'차이':>8s}")
for v, n in [(0,'배경'), (1,'침엽수림'), (2,'활엽수림'), (3,'혼효림')]:
    m = (label == v)
    print(f'{n:10s}{np.nanmedian(ndvi_s[m]):8.3f}'
          f'{np.nanmedian(ndvi_w[m]):9.3f}{np.nanmedian(dndvi[m]):8.3f}')

<IPython.core.display.HTML object>

[그림/지도 출력 생략 — 파일 경량화]


클래스             여름      낙엽기      차이
배경           0.820    0.462   0.309
침엽수림         0.852    0.646   0.198
활엽수림         0.885    0.438   0.438
혼효림          0.873    0.539   0.316


복구 restore


In [ ]:
# ============================================================
# [RESTORE] 전체 복구 (런타임 끊길 때 이 셀 하나만)
#   첫 실행: 피처 계산 + 모델 학습 후 캐시 저장   (약 9분)
#   이후   : 캐시와 모델을 로드만 함              (약 1분)
#   캐시를 새로 만들려면 REBUILD = True 로 두고 실행
# ============================================================
!pip install -q rasterio

import numpy as np, rasterio, os, gc, joblib
from google.colab import drive
from scipy.ndimage import uniform_filter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

REBUILD = False          # True 로 바꾸면 캐시 무시하고 다시 계산

drive.mount('/content/drive')
D      = '/content/drive/MyDrive/forest/data'
OUT    = '/content/drive/MyDrive/forest/outputs'
os.makedirs(OUT, exist_ok=True)
S2_PATH = f'{D}/s2/daejeon_s2_20band.tif'
CACHE   = f'{D}/daejeon_feats38.npy'
MODEL   = f'{OUT}/rf3_38feat_fresh2024.pkl'

# ---- 라벨 ----
with rasterio.open(f'{D}/daejeon_label.tif') as s: label = s.read(1)
with rasterio.open(f'{D}/daejeon_year.tif')  as s: year  = s.read(1)
shape  = label.shape
rng    = np.random.default_rng(42)
names  = ['침엽수림', '활엽수림', '혼효림']
C      = {1: 55.0, 2: 65.0, 3: 60.0}     # 임시 탄소계수 (교체 예정)
PIX_HA = 0.01
print('라벨 로드', shape)

# ---- 공간 분할 (시드 고정 → 항상 동일 분할) ----
TILE = 500
tile_id = (np.arange(shape[0])//TILE)[:,None]*1000 + (np.arange(shape[1])//TILE)[None,:]
uniq = np.unique(tile_id); rng.shuffle(uniq)
is_train = np.isin(tile_id, list(uniq[:int(len(uniq)*0.7)]))
valid    = (label >= 1) & (label <= 3)
print('분할 | 학습', (valid&is_train).sum(), '평가', (valid&~is_train).sum())

# ---- 피처 38개 (캐시) ----
if os.path.exists(CACHE) and not REBUILD:
    feats2 = np.load(CACHE).astype('float32')
    print('캐시에서 로드')
else:
    with rasterio.open(S2_PATH) as s: X = s.read().astype('float32')
    nd = lambda a, b: (a-b)/(a+b+1e-6)
    s_ndvi, w_ndvi = nd(X[6], X[2]),  nd(X[16], X[12])
    s_ndmi, w_ndmi = nd(X[6], X[8]),  nd(X[16], X[18])
    feats = np.concatenate([X, np.stack([s_ndvi, w_ndvi, s_ndmi, w_ndmi,
                                         s_ndvi-w_ndvi, s_ndmi-w_ndmi])])
    del X

    def ctx(a, w):
        a  = np.nan_to_num(a)
        mu = uniform_filter(a, size=w)
        sd = np.sqrt(np.maximum(uniform_filter(a**2, size=w) - mu**2, 0))
        return mu, sd

    extra = []
    for i in (20, 21, 24):            # s_ndvi, w_ndvi, d_ndvi
        for w in (5, 15):             # 50m, 150m
            extra += list(ctx(feats[i], w))
    feats2 = np.concatenate([feats, np.stack(extra).astype('float32')])
    np.save(CACHE, feats2.astype('float16'))
    del feats, extra; gc.collect()
    print('계산 후 캐시 저장')

F2 = feats2.reshape(feats2.shape[0], -1).T
y  = label.ravel()
del feats2; gc.collect()
print('피처', F2.shape[1], '개')

# ---- 모델 (캐시) ----
valid_f = valid & (year >= 2024)
tr_f = rng.choice(np.where((valid_f &  is_train).ravel())[0], 200_000, replace=False)
te_f = rng.choice(np.where((valid_f & ~is_train).ravel())[0], 300_000, replace=False)

if os.path.exists(MODEL) and not REBUILD:
    rf3 = joblib.load(MODEL)
    print('모델 로드')
else:
    rf3 = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                 n_jobs=-1, class_weight='balanced',
                                 random_state=42)
    rf3.fit(np.nan_to_num(F2[tr_f]), y[tr_f])
    joblib.dump(rf3, MODEL, compress=3)
    print('모델 학습 후 저장')

pred3 = rf3.predict(np.nan_to_num(F2[te_f]))
print(classification_report(y[te_f], pred3, target_names=names, digits=3))
print('macro F1:', round(f1_score(y[te_f], pred3, average='macro'), 3), '(기대 0.628)')
print('복구 완료')

Mounted at /content/drive
라벨 로드 (3506, 2813)
분할 | 학습 1840101 평가 765125
캐시에서 로드
피처 38 개
모델 로드
              precision    recall  f1-score   support

        침엽수림      0.762     0.761     0.761    108449
        활엽수림      0.774     0.790     0.782    137000
         혼효림      0.350     0.333     0.341     54551

    accuracy                          0.696    300000
   macro avg      0.629     0.628     0.628    300000
weighted avg      0.693     0.696     0.694    300000

macro F1: 0.628 (기대 0.628)
복구 완료


### `[K]` 공간 분할픽셀 랜덤 분할은 옆 픽셀이 학습셋에 들어가 성능을 부풀립니다. 500px 타일 단위로 나눕니다.

---## 모델픽셀 랜덤 분할은 옆 픽셀이 학습셋에 들어가 성능을 부풀립니다.**500픽셀 타일 단위**로 나눕니다.

In [ ]:
# ============================================================
# [K] 공간 분할
#   - 픽셀 랜덤 분할은 옆 픽셀이 학습셋에 들어가 성능을 부풀린다
#   - 5km 타일로 쪼개 타일 단위로 train/test 배정
#   - 학습 대상은 산림 3클래스(1,2,3)만. 배경/nodata 제외
# ============================================================
TILE = 500                    # 500px × 10m = 5km

rows = np.arange(shape[0]) // TILE
cols = np.arange(shape[1]) // TILE
tile_id = rows[:, None] * 1000 + cols[None, :]

uniq = np.unique(tile_id)
rng.shuffle(uniq)
n_tr = int(len(uniq) * 0.7)
train_tiles = set(uniq[:n_tr].tolist())

is_train = np.isin(tile_id, list(train_tiles))
valid    = (label >= 1) & (label <= 3)

print(f'타일 {len(uniq)}개 → 학습 {n_tr} / 평가 {len(uniq)-n_tr}')
print(f'학습 픽셀 {(valid &  is_train).sum():>10,}')
print(f'평가 픽셀 {(valid & ~is_train).sum():>10,}')

for v, n in [(1,'침엽'), (2,'활엽'), (3,'혼효')]:
    a = (label == v) & valid & is_train
    b = (label == v) & valid & ~is_train
    print(f'  {n} 학습 {a.sum():>9,} / 평가 {b.sum():>9,}')

타일 48개 → 학습 33 / 평가 15
학습 픽셀  1,840,101
평가 픽셀    765,125
  침엽 학습   634,959 / 평가   254,930
  활엽 학습   866,565 / 평가   385,687
  혼효 학습   338,577 / 평가   124,508


In [ ]:
# ============================================================
# [L] 피처 생성
#   - 원본 20밴드 + 식생지수 6개
#   - 밴드 순서: 0~9 여름(B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12)
#                10~19 낙엽기 (동일 순서)
#   - 어제 확인된 핵심 신호(dNDVI)를 명시적으로 넣어준다
# ============================================================
with rasterio.open(S2_PATH) as src:
    X = src.read().astype('float32')
    print('밴드명:', src.descriptions)

def nd(a, b):
    return (a - b) / (a + b + 1e-6)

s_ndvi = nd(X[6],  X[2])      # 여름   B8, B4
w_ndvi = nd(X[16], X[12])     # 낙엽기 B8, B4
s_ndmi = nd(X[6],  X[8])      # 여름   B8, B11 (수분)
w_ndmi = nd(X[16], X[18])     # 낙엽기
d_ndvi = s_ndvi - w_ndvi      # 핵심 신호
d_ndmi = s_ndmi - w_ndmi

feats = np.concatenate([X, np.stack([s_ndvi, w_ndvi, s_ndmi,
                                     w_ndmi, d_ndvi, d_ndmi])])
del X
print('총 피처 수:', feats.shape[0])

밴드명: ('s_B2', 's_B3', 's_B4', 's_B5', 's_B6', 's_B7', 's_B8', 's_B8A', 's_B11', 's_B12', 'w_B2', 'w_B3', 'w_B4', 'w_B5', 'w_B6', 'w_B7', 'w_B8', 'w_B8A', 'w_B11', 'w_B12')
총 피처 수: 26


In [ ]:
# ============================================================
# [M] Random Forest 베이스라인
#   - 메모리 절약을 위해 학습 20만 / 평가 30만 픽셀 서브샘플링
#   - 주지표는 macro F1 (클래스별 F1의 평균)
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

F = feats.reshape(feats.shape[0], -1).T
y = label.ravel()

tr_idx = np.where((valid &  is_train).ravel())[0]
te_idx = np.where((valid & ~is_train).ravel())[0]
tr_idx = rng.choice(tr_idx, min(200_000, len(tr_idx)), replace=False)
te_idx = rng.choice(te_idx, min(300_000, len(te_idx)), replace=False)

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                            n_jobs=-1, class_weight='balanced',
                            random_state=42)
rf.fit(np.nan_to_num(F[tr_idx]), y[tr_idx])

pred = rf.predict(np.nan_to_num(F[te_idx]))
names = ['침엽수림', '활엽수림', '혼효림']

print(classification_report(y[te_idx], pred, target_names=names, digits=3))
print('macro F1:', round(f1_score(y[te_idx], pred, average='macro'), 3))
print('\n혼동행렬 (행=정답, 열=예측)')
print(confusion_matrix(y[te_idx], pred))

              precision    recall  f1-score   support

        침엽수림      0.740     0.729     0.734     99970
        활엽수림      0.757     0.839     0.796    151252
         혼효림      0.301     0.210     0.247     48778

    accuracy                          0.700    300000
   macro avg      0.599     0.592     0.592    300000
weighted avg      0.677     0.700     0.686    300000

macro F1: 0.592

혼동행렬 (행=정답, 열=예측)
[[ 72830  15532  11608]
 [ 12172 126921  12159]
 [ 13365  25188  10225]]


In [ ]:
# ============================================================
# [N] 공간 컨텍스트 피처 추가
#   가설: 혼효림은 임분 단위 성질이므로, 주변 픽셀의
#         평균/분산을 보면 구분이 개선될 것이다
#   - 이동창 평균: 일대의 전반적 특성
#   - 이동창 표준편차: 섞여 있으면 값이 커진다 (핵심)
# ============================================================
from scipy.ndimage import uniform_filter

def ctx(a, w):
    """윈도우 w에서의 평균과 표준편차"""
    a = np.nan_to_num(a)
    mu  = uniform_filter(a,    size=w)
    mu2 = uniform_filter(a**2, size=w)
    sd  = np.sqrt(np.maximum(mu2 - mu**2, 0))
    return mu, sd

extra = []
for i, nm in [(20,'s_ndvi'), (21,'w_ndvi'), (24,'d_ndvi')]:
    for w in (5, 15):          # 50m, 150m 반경
        mu, sd = ctx(feats[i], w)
        extra += [mu, sd]

feats2 = np.concatenate([feats, np.stack(extra).astype('float32')])
print('피처 수:', feats.shape[0], '→', feats2.shape[0])

피처 수: 26 → 38


In [ ]:
# ============================================================
# [O] 컨텍스트 피처로 재학습 (동일 조건 비교)
#   - 분할, 시드, 하이퍼파라미터 모두 동일하게 유지
#   - 오직 피처만 다름
# ============================================================
F2 = feats2.reshape(feats2.shape[0], -1).T

rf2 = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                             n_jobs=-1, class_weight='balanced',
                             random_state=42)
rf2.fit(np.nan_to_num(F2[tr_idx]), y[tr_idx])
pred2 = rf2.predict(np.nan_to_num(F2[te_idx]))

print(classification_report(y[te_idx], pred2, target_names=names, digits=3))
print('macro F1:', round(f1_score(y[te_idx], pred2, average='macro'), 3),
      '  (기존 0.592)')
print(confusion_matrix(y[te_idx], pred2))

              precision    recall  f1-score   support

        침엽수림      0.761     0.723     0.741     99970
        활엽수림      0.761     0.850     0.803    151252
         혼효림      0.324     0.239     0.275     48778

    accuracy                          0.708    300000
   macro avg      0.615     0.604     0.607    300000
weighted avg      0.690     0.708     0.697    300000

macro F1: 0.607   (기존 0.592)
[[ 72260  15392  12318]
 [ 10649 128584  12019]
 [ 12034  25068  11676]]


In [ ]:
# ============================================================
# [P] 라벨 시의성 실험
#   가설: 오래된 갱신년도 폴리곤이 라벨 노이즈로 작용해
#         성능을 끌어내리고 있을 것이다
#   - 학습/평가 모두 2024년 이후 폴리곤으로 제한
#   - 다른 조건(분할, 시드, 하이퍼파라미터)은 전부 동일
# ============================================================
fresh = (year >= 2024)

print('갱신년도 분포 (산림 픽셀 기준)')
for v, n in [(1,'침엽'), (2,'활엽'), (3,'혼효')]:
    m = (label == v)
    print(f'  {n}  전체 {m.sum():>9,} / 2024+ {(m & fresh).sum():>9,}'
          f'  ({(m & fresh).sum()*100/m.sum():4.1f}%)')

valid_f = valid & fresh
tr_f = np.where((valid_f &  is_train).ravel())[0]
te_f = np.where((valid_f & ~is_train).ravel())[0]
print(f'\n학습 가능 {len(tr_f):,} / 평가 가능 {len(te_f):,}')

tr_f = rng.choice(tr_f, min(200_000, len(tr_f)), replace=False)
te_f = rng.choice(te_f, min(300_000, len(te_f)), replace=False)

rf3 = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                             n_jobs=-1, class_weight='balanced',
                             random_state=42)
rf3.fit(np.nan_to_num(F2[tr_f]), y[tr_f])
pred3 = rf3.predict(np.nan_to_num(F2[te_f]))

print(classification_report(y[te_f], pred3, target_names=names, digits=3))
print('macro F1:', round(f1_score(y[te_f], pred3, average='macro'), 3),
      '  (전체 라벨 0.607)')
print(confusion_matrix(y[te_f], pred3))

갱신년도 분포 (산림 픽셀 기준)
  침엽  전체   889,889 / 2024+   410,192  (46.1%)
  활엽  전체 1,252,252 / 2024+   410,816  (32.8%)
  혼효  전체   463,085 / 2024+   208,592  (45.0%)

학습 가능 712,827 / 평가 가능 316,773
              precision    recall  f1-score   support

        침엽수림      0.759     0.762     0.761    108371
        활엽수림      0.776     0.791     0.783    137034
         혼효림      0.350     0.331     0.340     54595

    accuracy                          0.697    300000
   macro avg      0.628     0.628     0.628    300000
weighted avg      0.692     0.697     0.694    300000

macro F1: 0.628   (전체 라벨 0.607)
[[ 82615  10834  14922]
 [ 10153 108336  18545]
 [ 16028  20507  18060]]


### `[P2]` 라벨 시의성 — 평가셋 고정판 ★기존 `[P]`는 학습·평가를 둘 다 바꿔 교란됐습니다. 여기서는 **평가셋을 2024+ 로 고정하고 학습셋만** 바꿉니다.결과: 침엽·활엽은 재현 변동 범위, **혼효림만 0.245 → 0.342.** 라벨 노후화 효과는 혼효림에만 있습니다.

### `[P2]` 라벨 시의성 — 평가셋 고정판기존 `[P]`는 학습과 평가를 **둘 다** 바꿔 교란됐습니다.여기서는 평가셋을 2024+ 로 고정하고 **학습셋만** 바꿉니다.

In [ ]:
# ============================================================
# [P2] 라벨 시의성 실험 — 평가셋 고정판 (기존 [P] 교란 제거)
#
#   기존 [P] 의 문제: 학습·평가 모두 2024+ 로 바꿔 두 효과가 섞였다.
#     ① 학습 라벨이 신선해짐   ② 평가 라벨/클래스 구성이 바뀜
#   macro F1 0.607→0.628 중 활엽은 오히려 −0.020, accuracy 는 −0.011.
#   상승분 대부분이 혼효림(+0.065) 하나에서 나왔다.
#
#   이 셀: 평가셋을 2024+ 로 고정하고 학습셋만 A/B/C 로 바꾼다.
#     A 전체 라벨      B 2024+ 만      C 2024 미만만 (역대조)
#   학습 픽셀 수·시드·하이퍼파라미터·타일분할 전부 동일.
#
#   판정: B > A > C 이고 B−A 가 재현 변동(macro F1 0.005)을 넘으면
#         "오래된 라벨이 학습을 해친다" 성립.
#         B ≈ A 면 기존 [P] 의 상승은 평가셋 교란이었다는 뜻.
#
#   필요 변수 ([RESTORE] 가 복구): F2, y, valid, is_train, year, rng, names
# ============================================================
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report

SEED, N_TR, N_TE = 42, 200_000, 300_000
fresh = (year >= 2024)

# ---- 평가셋 고정: 2024+ 이고 평가 타일인 픽셀 ----
te_pool = np.where((valid & fresh & ~is_train).ravel())[0]
rng_e   = np.random.default_rng(SEED)
te      = rng_e.choice(te_pool, min(N_TE, len(te_pool)), replace=False)
print(f'고정 평가셋 {len(te):,}  (풀 {len(te_pool):,})')
print('  클래스 구성  ' + '  '.join(
    f'{n} {(y[te]==v).sum()*100/len(te):4.1f}%' for v, n in zip((1,2,3), names)))

# ---- 학습셋 3종 ----
POOLS = {
    'A 전체 라벨':   valid & ~np.zeros_like(valid),
    'B 2024+ 만':    valid & fresh,
    'C 2024 미만만': valid & ~fresh,
}

print(f'\n{"학습셋":14s}{"풀":>12s}{"침엽":>7s}{"활엽":>7s}{"혼효":>7s}'
      f'{"macro":>8s}{"acc":>7s}')
RES = {}
for tag, pool in POOLS.items():
    tr_pool = np.where((pool & is_train).ravel())[0]
    if len(tr_pool) < N_TR:
        print(f'{tag:14s}{len(tr_pool):12,}  ← 픽셀 부족, 건너뜀')
        continue
    rng_t = np.random.default_rng(SEED)          # 세 조건 동일 시드
    tr = rng_t.choice(tr_pool, N_TR, replace=False)

    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                n_jobs=-1, class_weight='balanced',
                                random_state=SEED)
    rf.fit(np.nan_to_num(F2[tr]), y[tr])
    pr = rf.predict(np.nan_to_num(F2[te]))

    f1c = f1_score(y[te], pr, average=None, labels=[1, 2, 3])
    mac = f1_score(y[te], pr, average='macro')
    acc = accuracy_score(y[te], pr)
    RES[tag] = (f1c, mac, acc)
    print(f'{tag:14s}{len(tr_pool):12,}'
          + ''.join(f'{v:7.3f}' for v in f1c) + f'{mac:8.3f}{acc:7.3f}')

# ---- 판정 ----
NOISE_F1 = 0.005
if 'A 전체 라벨' in RES and 'B 2024+ 만' in RES:
    dBA = RES['B 2024+ 만'][1] - RES['A 전체 라벨'][1]
    print(f'\nB − A = {dBA:+.3f}   재현 변동 {NOISE_F1}   '
          f'→ {abs(dBA)/NOISE_F1:.1f}배')
    if dBA > 2 * NOISE_F1:
        print('★ 신선한 라벨로 학습하면 유의하게 낫다. '
              '"오래된 라벨이 노이즈" 성립 → 서론 근거로 사용 가능')
    elif abs(dBA) <= 2 * NOISE_F1:
        print('★ 차이 없음. 기존 [P] 의 0.607→0.628 은 평가셋 교란이었다.')
        print('  서론은 라벨 갱신 현황(67.1%, 지역별 편차)만으로 쓰고,')
        print('  "오래된 라벨이 해롭다"는 주장은 하지 말 것')
    else:
        print('★ 오히려 악화. 데이터 양 효과가 라벨 품질보다 크다는 뜻')
if 'C 2024 미만만' in RES:
    dBC = RES['B 2024+ 만'][1] - RES['C 2024 미만만'][1]
    print(f'B − C = {dBC:+.3f}  (역대조. A/B 차이보다 크게 나와야 일관)')

print('\n=== 상세 (B 2024+ 학습) ===')
if 'B 2024+ 만' in RES:
    tr_pool = np.where(((valid & fresh) & is_train).ravel())[0]
    tr = np.random.default_rng(SEED).choice(tr_pool, N_TR, replace=False)
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                n_jobs=-1, class_weight='balanced',
                                random_state=SEED).fit(
        np.nan_to_num(F2[tr]), y[tr])
    print(classification_report(y[te], rf.predict(np.nan_to_num(F2[te])),
                                target_names=names, digits=3))

고정 평가셋 300,000  (풀 316,773)
  클래스 구성  침엽수림 36.2%  활엽수림 45.7%  혼효림 18.2%

학습셋                      풀     침엽     활엽     혼효   macro    acc
A 전체 라벨          1,840,101  0.765  0.793  0.299   0.619  0.706
B 2024+ 만          712,827  0.762  0.784  0.342   0.629  0.698
C 2024 미만만       1,127,274  0.765  0.793  0.245   0.601  0.707

B − A = +0.010   재현 변동 0.005   → 2.0배
★ 신선한 라벨로 학습하면 유의하게 낫다. "오래된 라벨이 노이즈" 성립 → 서론 근거로 사용 가능
B − C = +0.028  (역대조. A/B 차이보다 크게 나와야 일관)

=== 상세 (B 2024+ 학습) ===
              precision    recall  f1-score   support

        침엽수림      0.762     0.761     0.762    108499
        활엽수림      0.774     0.794     0.784    137004
         혼효림      0.353     0.331     0.342     54497

    accuracy                          0.698    300000
   macro avg      0.630     0.629     0.629    300000
weighted avg      0.693     0.698     0.695    300000



대전 편향 진단

### `[Q]` 경계 픽셀 제외 — 혼효림의 천장경계를 빼면 침엽·활엽은 0.84~0.86까지 오르는데 **혼효만 0.376에서 멈춥니다.**혼효림은 픽셀이 아니라 임분(숲 덩어리)의 성질이라 픽셀 단위로는 원리적 한계가 있습니다. → U-Net 명분.

---## 오차 분석분류 성능에서 그치지 않고 **최종 산출물인 탄소축적량 오차**까지 따라갑니다.

In [ ]:
# ============================================================
# [Q] 파편화 진단
#   질문: 혼효림 성능이 나쁜 것이 대전의 도시적 파편화 탓인가?
#   방법: 경계 픽셀(3x3 이웃에 다른 클래스 존재)을 제외하고 재평가
#         내부 픽셀만으로 성능이 크게 오르면 파편화 영향이 크다는 뜻
# ============================================================
from scipy.ndimage import maximum_filter, minimum_filter

lab_f = label.astype('int16')
lab_f[label == 255] = -1
is_edge = (maximum_filter(lab_f, 3) != minimum_filter(lab_f, 3))

print('클래스별 경계 픽셀 비율')
for v, n in [(1,'침엽'), (2,'활엽'), (3,'혼효')]:
    m = (label == v)
    print(f'  {n}  {(m & is_edge).sum()*100/m.sum():5.1f}%')

# 갱신년도 필터 + 경계 제외 (P 조건에 경계만 추가)
core  = valid & (year >= 2024) & ~is_edge
te_c  = np.where((core & ~is_train).ravel())[0]
te_c  = rng.choice(te_c, min(300_000, len(te_c)), replace=False)

pred_c = rf3.predict(np.nan_to_num(F2[te_c]))
print()
print(classification_report(y[te_c], pred_c, target_names=names, digits=3))
print('내부 픽셀만 macro F1:',
      round(f1_score(y[te_c], pred_c, average='macro'), 3), '  (P: 0.628)')

클래스별 경계 픽셀 비율
  침엽   45.4%
  활엽   36.0%
  혼효   48.3%

              precision    recall  f1-score   support

        침엽수림      0.841     0.838     0.840     62817
        활엽수림      0.849     0.869     0.859     90029
         혼효림      0.390     0.363     0.376     26842

    accuracy                          0.783    179688
   macro avg      0.693     0.690     0.692    179688
weighted avg      0.778     0.783     0.780    179688

내부 픽셀만 macro F1: 0.692   (P: 0.628)


In [ ]:
# ============================================================
# [T] 면적 → 탄소축적량 오차 전파 (1차 시도)
#   - 분류 오차가 임상별 면적 오차로, 다시 탄소량 오차로
#     어떻게 증폭되는지 정량화
#   - 이 프로젝트의 최종 목표에 대한 첫 답
#
#   ※ 탄소계수는 임시값. 국립산림과학원 공식 계수로 교체 필요 (박유경)
# ============================================================
CARBON = {1: 55.0, 2: 65.0, 3: 60.0}   # tC/ha  ← TODO: 실제 계수로 교체
PIX_HA = 0.01                          # 10m x 10m = 100m² = 0.01ha

# 평가 지역 전체를 예측 (샘플링 없이)
mask_eval = valid & (year >= 2024) & ~is_train
idx_all   = np.where(mask_eval.ravel())[0]
pred_all  = rf3.predict(np.nan_to_num(F2[idx_all]))
true_all  = y[idx_all]

print(f'{"클래스":8s}{"실제(ha)":>12s}{"예측(ha)":>12s}{"면적오차":>10s}')
tot_t = tot_p = 0
for v, n in [(1,'침엽'), (2,'활엽'), (3,'혼효')]:
    a_t = (true_all == v).sum() * PIX_HA
    a_p = (pred_all == v).sum() * PIX_HA
    tot_t += a_t * CARBON[v]
    tot_p += a_p * CARBON[v]
    print(f'{n:8s}{a_t:12,.0f}{a_p:12,.0f}{(a_p-a_t)*100/a_t:9.1f}%')

print(f'\n탄소축적량 실제 {tot_t:,.0f} tC')
print(f'탄소축적량 예측 {tot_p:,.0f} tC')
print(f'오차 {(tot_p-tot_t)*100/tot_t:+.1f}%')

클래스           실제(ha)      예측(ha)      면적오차
침엽             1,145       1,143     -0.2%
활엽             1,447       1,477      2.1%
혼효               576         548     -4.9%

탄소축적량 실제 191,576 tC
탄소축적량 예측 191,734 tC
오차 +0.1%


In [ ]:
# ============================================================
# [T2] 탄소계수 민감도 분석
#   질문: 탄소량 오차 +0.1%는 계수 설정에 얼마나 의존하는가?
#   방법: 침엽/활엽 계수 격차를 여러 시나리오로 바꿔가며 오차 재계산
#   - 격차가 커질수록 오차가 증폭되면, 정확한 계수 확보가 필수라는 뜻
# ============================================================
area_t = {v: (true_all == v).sum() * PIX_HA for v in (1,2,3)}
area_p = {v: (pred_all == v).sum() * PIX_HA for v in (1,2,3)}

scenarios = {
    '임시값 (55/65/60)'      : {1:55, 2:65, 3:60},
    '격차 작음 (58/62/60)'   : {1:58, 2:62, 3:60},
    '격차 큼 (45/75/60)'     : {1:45, 2:75, 3:60},
    '격차 매우 큼 (40/85/62)': {1:40, 2:85, 3:62},
    '혼효=침엽쪽 (55/65/56)' : {1:55, 2:65, 3:56},
    '혼효=활엽쪽 (55/65/64)' : {1:55, 2:65, 3:64},
}

print(f'{"시나리오":24s}{"실제(tC)":>12s}{"예측(tC)":>12s}{"오차":>9s}')
for nm, C in scenarios.items():
    t = sum(area_t[v]*C[v] for v in (1,2,3))
    p = sum(area_p[v]*C[v] for v in (1,2,3))
    print(f'{nm:24s}{t:12,.0f}{p:12,.0f}{(p-t)*100/t:8.2f}%')

시나리오                          실제(tC)      예측(tC)       오차
임시값 (55/65/60)               191,576     191,734    0.08%
격차 작음 (58/62/60)             190,669     190,732    0.03%
격차 큼 (45/75/60)              194,599     195,076    0.24%
격차 매우 큼 (40/85/62)           204,498     205,227    0.36%
혼효=침엽쪽 (55/65/56)            189,271     189,542    0.14%
혼효=활엽쪽 (55/65/64)            193,880     193,927    0.02%


In [ ]:
# ============================================================
# [T3] 타일별 탄소 오차 분포
#   질문: 총량 오차 0.1%는 우연한 상쇄인가, 구조적으로 안정적인가?
#   방법: 평가 타일 각각에서 탄소 오차를 따로 계산해 분포를 본다
#   - 타일별 오차가 크게 흩어지면 총량 정확도는 착시
# ============================================================
C = {1: 55.0, 2: 65.0, 3: 60.0}

tile_flat = tile_id.ravel()[idx_all]
errs = []

for t in np.unique(tile_flat):
    m = (tile_flat == t)
    if m.sum() < 5000:            # 표본 적은 타일 제외
        continue
    ct = sum((true_all[m] == v).sum() * PIX_HA * C[v] for v in (1,2,3))
    cp = sum((pred_all[m] == v).sum() * PIX_HA * C[v] for v in (1,2,3))
    errs.append((cp - ct) * 100 / ct)

errs = np.array(errs)
print(f'타일 {len(errs)}개')
print(f'  평균   {errs.mean():+.2f}%')
print(f'  표준편차 {errs.std():.2f}%')
print(f'  최소   {errs.min():+.2f}%')
print(f'  최대   {errs.max():+.2f}%')
print(f'  절대오차 중앙값 {np.median(np.abs(errs)):.2f}%')
print('\n타일별 오차:', np.round(np.sort(errs), 2))

타일 9개
  평균   +0.02%
  표준편차 1.09%
  최소   -1.29%
  최대   +1.84%
  절대오차 중앙값 1.02%

타일별 오차: [-1.29 -1.13 -1.02 -0.56 -0.04  0.08  0.73  1.55  1.84]


### `[U]` 동서 블록 분할 ★★ — 보고서 7-1의 근거**같은 대전, 같은 데이터, 같은 모델인데 평가 영역을 공간적으로 떼어놓기만** 합니다.랜덤 타일 +0.08% → **동서 블록 +0.77%. 10배.**→ 9칸 전이 표의 "지역 내" 칸이 인접 타일 덕을 보고 있다는 직접 증거입니다. 배율 주장을 보류한 이유.

### `[U]` 동서 블록 분할 — 공간 분리만으로 오차가 커지는지랜덤 타일 분할은 학습 타일과 평가 타일이 **인접할 수 있습니다.**서쪽으로 학습해 동쪽에서 평가하면 학습에 전혀 안 쓴 연속된 땅에서 시험하는 셈입니다.→ **지역을 옮겼을 때 어떻게 될지의 예고편.**

In [ ]:
# ============================================================
# [U] 동서 블록 분할 — 더 엄격한 일반화 검증
#   랜덤 타일 분할은 학습/평가 타일이 서로 인접할 수 있다.
#   서쪽 절반으로 학습해 동쪽 절반에서 평가하면
#   학습에 전혀 안 쓴 연속된 땅에서 시험하는 셈이 된다.
#   → 다른 지역으로 옮겼을 때 어떻게 될지의 예고편
# ============================================================
mid = shape[1] // 2
is_train_geo = np.zeros(shape, bool)
is_train_geo[:, :mid] = True                 # 서쪽 = 학습

vf   = valid & (year >= 2024)
tr_g = np.where((vf &  is_train_geo).ravel())[0]
te_g = np.where((vf & ~is_train_geo).ravel())[0]
print(f'서쪽 학습 {len(tr_g):,} / 동쪽 평가 {len(te_g):,}')

rng_u = np.random.default_rng(7)             # RESTORE의 rng를 건드리지 않도록 별도 사용
tr_g  = rng_u.choice(tr_g, min(200_000, len(tr_g)), replace=False)
te_g  = rng_u.choice(te_g, min(300_000, len(te_g)), replace=False)

rf_g = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                              n_jobs=-1, class_weight='balanced',
                              random_state=42)
rf_g.fit(np.nan_to_num(F2[tr_g]), y[tr_g])
pg = rf_g.predict(np.nan_to_num(F2[te_g]))

print(classification_report(y[te_g], pg, target_names=names, digits=3))
print('동서분할 macro F1:', round(f1_score(y[te_g], pg, average='macro'), 3),
      '  (랜덤타일 0.628)')

ct = sum((y[te_g] == v).sum() * PIX_HA * C[v] for v in (1,2,3))
cp = sum((pg      == v).sum() * PIX_HA * C[v] for v in (1,2,3))
print(f'탄소 오차 {(cp-ct)*100/ct:+.2f}%  (랜덤타일 +0.08%)')

서쪽 학습 481,469 / 동쪽 평가 548,131
              precision    recall  f1-score   support

        침엽수림      0.717     0.750     0.733    114193
        활엽수림      0.658     0.840     0.738    119056
         혼효림      0.411     0.176     0.246     66751

    accuracy                          0.658    300000
   macro avg      0.595     0.589     0.572    300000
weighted avg      0.625     0.658     0.627    300000

동서분할 macro F1: 0.572   (랜덤타일 0.628)
탄소 오차 +0.77%  (랜덤타일 +0.08%)
